# Setup: Generate Sample Dataset

This cell creates the required folder structure (`data/raw/` and `data/processed/`) relative to the notebook, and generates the sample CSV dataset with missing values. 
This ensures the dataset is ready for cleaning functions and saves it to `data/raw/sample_data.csv`.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    ("src/cleaning.py", "NEEDED", "YOU write this in the homework - the import fails until you do"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/zhu/Documents/GitHub/bootcamp_wentao_zhu/homework/homework06

  [OK ]  NEEDED    src/cleaning.py                     YOU write this in the homework - the import fails until you do

All needed files present.


In [3]:
import os
import pandas as pd
import numpy as np

# Define folder paths relative to this notebook
raw_dir = 'data/raw'
processed_dir = 'data/processed'

# Create folders if they don't exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Define the sample data
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV in raw data folder
csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')


File already exists at data/raw/sample_data.csv. Skipping CSV creation to avoid overwrite.


# Homework Starter — Stage 6: Data Preprocessing
Use this notebook to apply your cleaning functions and save processed data.

In [4]:
import pandas as pd
from src import cleaning


## Load Raw Dataset

In [5]:
raw_df = pd.read_csv(
    'data/raw/sample_data.csv',
    dtype={'zipcode': 'string'},
)
df = raw_df.copy()
print('Raw shape:', df.shape)
print('Missing values before cleaning:')
display(df.isna().sum().to_frame('missing'))
df.head()


Raw shape: (7, 6)
Missing values before cleaning:


,missing
age,1
income,3
score,1
zipcode,0
city,0
extra_data,5


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


## Apply Cleaning Functions

In [6]:
# Remove columns that are mostly missing, fill remaining numeric gaps, then scale.
df = cleaning.drop_missing(df, threshold=0.5)
df = cleaning.fill_missing_median(df, ['age', 'income', 'score'])
df = cleaning.normalize_data(df, ['age', 'income', 'score'])

comparison = pd.DataFrame({
    'raw_missing': raw_df.isna().sum(),
    'clean_missing': df.isna().sum(),
})
print('Cleaned shape:', df.shape)
display(comparison.fillna('column dropped'))
display(df.describe(include='all').T)
df.head()


Cleaned shape: (7, 5)


,raw_missing,clean_missing
age,1,0.0
city,0,0.0
extra_data,5,column dropped
income,3,0.0
score,1,0.0
zipcode,0,0.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,7.0,NaN,NaN,NaN,0.5,0.328479,0.0,0.333333,0.5,0.666667,1.0
income,7.0,NaN,NaN,NaN,0.589286,0.314281,0.0,0.53125,0.625,0.71875,1.0
score,7.0,NaN,NaN,NaN,0.585165,0.325952,0.0,0.480769,0.596154,0.769231,1.0
zipcode,7,7,90210,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,7,7,Beverly,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin


## Save Cleaned Dataset and Assumptions

- Columns with more than 50% missing values are removed; this drops `extra_data`.
- Numeric gaps in `age`, `income`, and `score` are filled with the median because the sample is small and the median is resistant to extremes.
- The three numeric analysis columns are min-max normalized to the 0-1 range.
- `zipcode` remains text so leading zeros would not be lost, and city labels are retained as provided.


In [7]:
output_path = 'data/processed/sample_data_cleaned.csv'
df.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Remaining missing values:', int(df.isna().sum().sum()))


Saved: data/processed/sample_data_cleaned.csv
Remaining missing values: 0
